### Chargement des données

In [ ]:
import numpy as np
from shapely import wkb
from pyproj import Transformer
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa


fichier = 6
if fichier == 1:
    nom_fichier = 'Daniel_Joram'
    type_fichier = "parquet"
elif fichier == 2:
    nom_fichier = 'RDR2018 Alizés'
    type_fichier = "csv"
elif fichier == 3:
    nom_fichier = 'RDR2018 degolfage'
    type_fichier = "csv"
elif fichier == 4:
    nom_fichier = 'RDR2018 Sortie Manche'
    type_fichier = "csv"
elif fichier == 5:
    nom_fichier = 'Jef_Levasseur'
    type_fichier = "parquet"
elif fichier == 6:
    nom_fichier = 'Francois_Rucheton'
    type_fichier = "parquet"
elif fichier == 7:
    nom_fichier = 'Olivier_Costa'
    type_fichier = "parquet"
elif fichier == 8:
    nom_fichier = 'Daniel_Joram_2020'
    type_fichier = "parquet"
elif fichier == 9:
    nom_fichier = 'Tara_Ocean_Explorer'
    type_fichier = "parquet"
elif fichier == 10:
    nom_fichier = 'merge'
    type_fichier = "vela"


if type_fichier == "csv":
    data = pd.read_csv(f'../Data/{nom_fichier}.csv', encoding='cp1252', sep=';', low_memory=False)
elif type_fichier == "parquet" or type_fichier == "vela":
    data = pd.read_parquet(f'../Data/{nom_fichier}.parquet')


print(f'Nombre de données : {len(data)}')
data.head()
print(data.columns)


### Normalisation et Nettoyage

In [ ]:
def coord_to_decimal(value, axis):
    value = str(value).replace('Â', '').strip()
    if not value or value == 'nan':
        return None

    borne_max = 90 if axis == 'latitude' else 180

    parts = value.split()
    if len(parts) == 2 and '°' in parts[0]:
        degrees, minutes = parts[0].split('°')
        decimal = float(degrees) + float(minutes) / 60
        if parts[1] in ('S', 'W', 'W(T)', 'S(T)'):
            decimal *= -1
        return decimal

    texte = ''.join(char for char in value.replace(',', '.') if char.isdigit() or char in '.-')
    if texte in ('', '-', '.', '-.'):
        return None

    decimal = float(texte)
    while abs(decimal) > borne_max and abs(decimal) >= 100:
        decimal /= 10

    if abs(decimal) > borne_max:
        return None
    return decimal

def normaliser_valeur_vec(series, scale=1, zero_to_none=False):
    """Version vectorisée de normaliser_valeur (~50x plus rapide sur gros DataFrames)."""
    if pd.api.types.is_numeric_dtype(series):
        s = series.astype(float)
    else:
        s = pd.to_numeric(
            series.astype(str).str.replace(',', '.', regex=False)
                  .str.strip()
                  .str.extract(r'(-?\d+(?:\.\d+)?)', expand=False),
            errors='coerce'
        )
    if type_fichier == "parquet" and scale != 1:
        s = s / scale
    if zero_to_none:
        s = s.replace(0.0, float('nan'))
    return s

def add_date_seconds_column(
    df: pd.DataFrame,
    date_col="date TU",
    time_col="heure TU",
    output_col="Date"
):
    """
    Combine 'date TU' et 'heure TU' en une colonne Date (timestamp en secondes).
    Format attendu : 
        date TU  : '11/11/2018'
        heure TU : '14:24:17'

        OU

        date_col : None
        time_col : '08:34:53+00:00'
        source   : 'ForeSail_DarssHiddensee_020821.parquet' (jjmmaA)
    """

    df = df.copy()

    if date_col != None:

        # Combinaison date + heure puis conversion en datetime
        dt = pd.to_datetime(
            df[date_col] + " " + df[time_col],
            format="%d/%m/%Y %H:%M:%S",
            errors="coerce"
        )

        # Conversion en timestamp (secondes)
        df[output_col] = (dt.astype("int64") // 10**9).where(dt.notna(), np.nan)
    else:
        # Parse de l'heure/date (avec ou sans fuseau) en UTC
        t = pd.to_datetime(df[time_col], errors="coerce", utc=True)

        # Extrait la date depuis la colonne 'source' (ex: ..._020821.parquet)
        source_col = df.loc[:, "source"] if "source" in df.columns else pd.Series(index=df.index, dtype=object)
        if isinstance(source_col, pd.DataFrame):
            source_col = source_col.iloc[:, 0]
        source_name = source_col.fillna('').map(str).str.replace('\\', '/', regex=False).str.split('/').str[-1]
        # Prend les 6 derniers chiffres du nom (jjmmaa attendu)
        date_digits = source_name.str.extract(r'(\d{6})(?!.*\d)', expand=False)
        day = pd.to_numeric(date_digits.str.slice(0, 2), errors="coerce")
        month = pd.to_numeric(date_digits.str.slice(2, 4), errors="coerce")
        year = pd.to_numeric(date_digits.str.slice(4, 6), errors="coerce") + 2000
        date_source = pd.to_datetime(
            pd.DataFrame({"year": year, "month": month, "day": day}),
            errors="coerce"
        )

        # Construit datetime complet: date(source) + heure(timestamp)
        sec_day = t.dt.hour * 3600 + t.dt.minute * 60 + t.dt.second
        dt_from_source = date_source + pd.to_timedelta(sec_day, unit="s")

        # Fallback: si date source absente, conserve le datetime parsé
        dt = dt_from_source.fillna(t.dt.tz_convert("UTC").dt.tz_localize(None))

        # Conversion robuste en epoch secondes (évite les valeurs parasites)
        dt_utc = pd.to_datetime(dt, errors="coerce", utc=True)
        df[output_col] = (dt_utc - pd.Timestamp("1970-01-01", tz="UTC")) / pd.Timedelta(seconds=1)

    return df

nb_suppr = 0


if type_fichier == "csv":
    data.rename(columns={'X': 'longitude', 'Y': 'latitude', 'vitesse fond': 'BSP'}, inplace=True)
    data['longitude'] = data['longitude'].apply(lambda value: coord_to_decimal(value, 'longitude'))
    data['latitude'] = data['latitude'].apply(lambda value: coord_to_decimal(value, 'latitude'))
    data = add_date_seconds_column(data)
elif type_fichier == "parquet":
    
    # 1) Decoder la geometrie WKB -> objet Point (list comprehension, ~5x plus rapide que apply)
    geom_col = "geom"
    points = [wkb.loads(bytes(b)) if b is not None else None for b in data[geom_col]]

    # 2) Extraire X/Y bruts (dans ton cas probablement EPSG:3395)
    data["X_3395"] = [p.x if p else None for p in points]
    data["Y_3395"] = [p.y if p else None for p in points]

    # 3) Convertir EPSG:3395 -> WGS84 (longitude/latitude) en vectorise
    transformer = Transformer.from_crs("EPSG:3395", "EPSG:4326", always_xy=True)
    valid_mask = data["X_3395"].notna() & data["Y_3395"].notna()
    lons_out, lats_out = transformer.transform(
        data.loc[valid_mask, "X_3395"].values,
        data.loc[valid_mask, "Y_3395"].values
    )
    data["longitude"] = np.nan
    data["latitude"] = np.nan
    data.loc[valid_mask, "longitude"] = lons_out
    data.loc[valid_mask, "latitude"] = lats_out

    data.rename(columns={'TrueWindAngle': 'TWA', 'TrueWindSpeed': 'TWS', 'SpeedOverGround': 'BSP'}, inplace=True)
elif type_fichier == "vela":
    geom_col = "geometry"

    points = [wkb.loads(bytes(b)) if b is not None else None for b in data[geom_col]]

    # 2) Extraire X/Y bruts (dans ton cas probablement EPSG:3395)
    data["longitude"] = [p.x if p else None for p in points]
    data["latitude"] = [p.y if p else None for p in points]

    data.rename(columns={'wind_speed':'TWS', 'wind_angle': 'TWA', 'sog': 'BSP'}, inplace=True)
    data = add_date_seconds_column(data, date_col=None, time_col="timestamp")
    
data["TWA"] = normaliser_valeur_vec(data["TWA"], scale=10)
data["TWS"] = normaliser_valeur_vec(data["TWS"], scale=100, zero_to_none=True)
data["BSP"] = normaliser_valeur_vec(data["BSP"], scale=100)

# Suppression explicite des lignes avec TWS nul ou invalide.
new_data = data.dropna(subset=["TWS"])
new_data = new_data[new_data["TWS"] != 0]
new_data = new_data[new_data["BSP"] > 2]  # Suppression des valeurs aberrantes
nb_suppr += len(data) - len(new_data)
data = new_data.reset_index(drop=True)
print(f"Nombre total de points supprimés : {nb_suppr}")
print(f'Nombre de points restants : {len(data)}')
data.head()

### Suppression des outliers

In [ ]:
def haversine_vec(lat1, lon1, lat2, lon2):
    """Haversine vectorisé (numpy arrays) — tableau de distances en mètres."""
    R = 6_371_000.0
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(np.radians(lat1))
        * np.cos(np.radians(lat2))
        * np.sin(dlon / 2) ** 2
    )
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

def remove_outliers(
    df: pd.DataFrame,
    lat_col="latitude",
    lon_col="longitude",
    date_col="Date",
    distance_threshold=3000,  # mètres
    time_threshold=3600       # secondes
):
    """
    Supprime uniquement les points aberrants basés sur distance ou temps.
    
    Retourne :
    - cleaned_df : DataFrame nettoyé
    - removed_indices : indices supprimés (dans le DF original)
    """

    if len(df) < 2:
        return df.copy(), []

    df = df.reset_index(drop=False)  # conserver index original
    orig_index = df["index"].to_numpy()

    lats = df[lat_col].to_numpy()
    lons = df[lon_col].to_numpy()
    times = df[date_col].to_numpy().astype(np.float64)

    # Distances & temps entre points successifs
    dists = haversine_vec(lats[:-1], lons[:-1], lats[1:], lons[1:])
    dts = times[1:] - times[:-1]

    # Détection outliers
    is_outlier = (dists > distance_threshold) | (dts > time_threshold)

    # On supprime le point suivant (responsable de la rupture)
    outlier_idx = np.where(is_outlier)[0] + 1

    # Masque de conservation
    keep_mask = np.ones(len(df), dtype=bool)
    keep_mask[outlier_idx] = False

    cleaned_df = df.loc[keep_mask].drop(columns="index").reset_index(drop=True)
    removed_indices = orig_index[outlier_idx].tolist()

    return cleaned_df, removed_indices

data, removed_indices = remove_outliers(data)
nb_suppr += len(removed_indices)
print(f"Nombre total de points supprimés : {nb_suppr}")
print(f'Nombre de points restants : {len(data)}')

### Suppression des voyages inutiles

In [ ]:
def detect_trips(
    df: pd.DataFrame,
    lat_col="latitude",
    lon_col="longitude",
    date_col="Date",
    distance_threshold=5000,
    time_threshold=3600,
    min_points=100
):
    n = len(df)
    if n < min_points:
        return df.iloc[0:0], [], n

    df = df.reset_index(drop=True)

    lats = df[lat_col].values
    lons = df[lon_col].values
    times = df[date_col].values.astype(np.float64)

    dists = haversine_vec(lats[:-1], lons[:-1], lats[1:], lons[1:])
    dts = times[1:] - times[:-1]

    is_break = (dists > distance_threshold) | (dts > time_threshold)
    break_indices = np.where(is_break)[0] + 1

    seg_starts = np.concatenate([[0], break_indices])
    seg_ends   = np.concatenate([break_indices - 1, [n - 1]])

    trips_orig = []
    valid_chunks = []
    for start, end in zip(seg_starts, seg_ends):
        if end - start + 1 >= min_points:
            trips_orig.append((int(start), int(end)))
            valid_chunks.append(np.arange(start, end + 1))

    all_valid = np.concatenate(valid_chunks) if valid_chunks else np.array([], dtype=int)

    # Remapper les indices de trips vers le nouvel espace (cleaned_df re-indexé à 0)
    if valid_chunks:
        old_starts = np.array([t[0] for t in trips_orig])
        old_ends   = np.array([t[1] for t in trips_orig])
        new_starts = np.searchsorted(all_valid, old_starts)
        new_ends   = np.searchsorted(all_valid, old_ends)
        trips = list(zip(new_starts.tolist(), new_ends.tolist()))
    else:
        trips = []

    suppr = n - len(all_valid)
    cleaned_df = df.iloc[all_valid].reset_index(drop=True)

    return cleaned_df, trips, suppr

def is_trip_significant(
    df: pd.DataFrame,
    trip: tuple,
    lat_col="latitude",
    lon_col="longitude",
    min_radius=500  # mètres
):
    """
    Version optimisée O(1) par trajet.
    Vérifie si un trajet sort d'un cercle minimal.

    Retourne :
    - bool : trajet significatif ou non
    - float : distance max estimée (m)
    """

    start, end = trip
    segment = df.loc[start:end]

    if len(segment) < 2:
        return False, 0.0

    lat_min = segment[lat_col].min()
    lat_max = segment[lat_col].max()
    lon_min = segment[lon_col].min()
    lon_max = segment[lon_col].max()

    lat_center = (lat_min + lat_max) / 2
    lon_center = (lon_min + lon_max) / 2

    # Distances maximales estimées
    north_south = haversine_vec(lat_min, lon_center, lat_max, lon_center)
    east_west   = haversine_vec(lat_center, lon_min, lat_center, lon_max)
    diagonal    = haversine_vec(lat_min, lon_min, lat_max, lon_max)

    max_dist = max(north_south, east_west, diagonal) / 2

    return max_dist >= min_radius, max_dist

data, voyages_temp, nb_sup = detect_trips(data)
nb_suppr += nb_sup

voyages = []
keep_mask = np.ones(len(data), dtype=bool)

for trip in voyages_temp:
    ok, radius = is_trip_significant(data, trip, min_radius=800)
    if ok:
        voyages.append(trip)
    else:
        start, end = trip
        keep_mask[start:end + 1] = False
        nb_suppr += end - start + 1

cleaned_data = data[keep_mask].reset_index(drop=True)
# Ajouter colonne VOYAGE
cleaned_data["voyage"] = -1  # valeur par défaut (hors voyage)

for i, (start, end) in enumerate(voyages, start=1):
    cleaned_data.loc[start:end, "voyage"] = i

print(f'Nombre de voyages détectés : {len(voyages)}')
print(f'Nombre de points supprimés : {nb_suppr}')
print(f'Nombre de données restantes : {len(cleaned_data)}')
cleaned_data.head()

### Export des données nettoyées

In [ ]:
pq.write_table(pa.Table.from_pandas(cleaned_data[['Date', 'TWA', 'TWS', 'BSP', 'longitude', 'latitude', 'voyage']]), f'../Cleaned_Data/Cleaned_{nom_fichier}.parquet')
print(f'Fichier Cleaned_{nom_fichier}.parquet créé avec succès.')